# 02 · Joins

The warehouse is a star schema. Each fact table holds the numbers and an ID for each thing it refers to (`store_id`, `sku`, `date`), and the dimension tables hold the descriptions. A join puts them back together.

`v_sales_flat` in notebook 01 was a shortcut: it's these joins, written once. Here you write them yourself.

In [1]:
from _setup import q

## One join at a time

To get a business year onto a sale, join `fact_sales` to `dim_date` on the date. To get a store type, join `dim_store` on `store_id`. The letters after each table (`f`, `d`, `s`) are aliases, short names that say which table each column comes from.

In [2]:
q("""
SELECT s.store_type,
       ROUND(SUM(f.net_sales_gbp), 0) AS net_sales
FROM fact_sales f
JOIN dim_date d  ON f.date = d.full_date
JOIN dim_store s ON f.store_id = s.store_id
WHERE d.business_year = 2025
GROUP BY s.store_type
ORDER BY net_sales DESC
""")

,store_type,net_sales
0,High Street,"10,481,263"
1,Retail Park,"10,187,119"
2,Online,"7,134,377"
3,Shopping Centre,"5,534,832"
4,Outlet,"3,280,385"
5,Garden Centre,"1,344,416"
6,Department Store,"1,038,268"


## Two dimensions in one query

Join the product table as well, and you can split by product and channel at the same time.

In [3]:
q("""
SELECT p.major_product_group,
       s.channel,
       ROUND(SUM(f.net_sales_gbp), 0) AS net_sales
FROM fact_sales f
JOIN dim_date d    ON f.date = d.full_date
JOIN dim_store s   ON f.store_id = s.store_id
JOIN dim_product p ON f.sku = p.sku
WHERE d.business_year = 2025
GROUP BY p.major_product_group, s.channel
ORDER BY p.major_product_group, net_sales DESC
""")

,major_product_group,channel,net_sales
0,Accessories,Retail,"932,593"
1,Accessories,Online,"221,958"
2,Accessories,Concession,"157,435"
3,Camping & Equipment,Retail,"103,472"
4,Camping & Equipment,Online,"17,304"
...,...,...,...
16,Outerwear,Online,"2,336,690"
17,Outerwear,Concession,"1,542,225"
18,Tops,Retail,"2,591,420"
19,Tops,Online,"674,057"


## LEFT JOIN: finding what's missing

An ordinary (inner) join only keeps rows that match on both sides. A `LEFT JOIN` keeps every row from the left table, with blanks (`NULL`) where nothing matched. That makes it the tool for "which products never sold?".

This should match the CMP-11 warning on the Data Quality page.

In [4]:
q("""
SELECT COUNT(*) AS skus_never_sold
FROM dim_product p
LEFT JOIN (SELECT DISTINCT sku FROM fact_sales) sold ON p.sku = sold.sku
WHERE sold.sku IS NULL
""")

,skus_never_sold
0,143


## The fan-out trap

Joining two fact tables directly is the classic SQL mistake. A store has one footfall row per day but many sales lines per day, so joining them repeats each footfall row once for every sales line, and the footfall total comes out far too big.

In [5]:
q("""
-- WRONG: footfall gets counted once per sales line
SELECT ROUND(SUM(ff.footfall), 0) AS footfall_wrong
FROM fact_footfall ff
JOIN fact_sales f ON ff.store_id = f.store_id AND ff.date = f.date
JOIN dim_date d   ON ff.date = d.full_date
WHERE d.business_year = 2025
""")

,footfall_wrong
0,"21,090,382"


The fix is to add each fact up to the same grain first, then join the totals. A `WITH` clause (a CTE) names each step so the query reads top to bottom.

In [6]:
q("""
-- RIGHT: total each fact by store first, then join the totals
WITH footfall AS (
    SELECT ff.store_id, SUM(ff.footfall) AS footfall
    FROM fact_footfall ff
    JOIN dim_date d ON ff.date = d.full_date
    WHERE d.business_year = 2025
    GROUP BY ff.store_id
),
sales AS (
    SELECT f.store_id, SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d ON f.date = d.full_date
    WHERE d.business_year = 2025
    GROUP BY f.store_id
)
SELECT ROUND(SUM(footfall.footfall), 0) AS footfall_right,
       ROUND(SUM(sales.net_sales), 0)   AS net_sales
FROM footfall
JOIN sales ON footfall.store_id = sales.store_id
""")

,footfall_right,net_sales
0,"2,280,593","31,866,283"


## Challenges

**2.1** Net sales by market for BY25, using joins rather than `v_sales_flat`. The markets should add up to about £39M.

In [7]:
# your query here
# q("""
#     SELECT ...
# """)

**2.2** Net sales by gender and major product group for BY25.

In [8]:
# your query here
# q("""
#     SELECT ...
# """)

**2.3** Which promotion sold the most units in BY25? Join `dim_promo` and leave out full price (`promo_id = 'PROMO0000'`).

In [9]:
# your query here
# q("""
#     SELECT ...
# """)

**2.4** Store conversion rate by market for BY25, from `fact_footfall`: total transactions divided by total footfall.

*Hint: this one only needs `fact_footfall`, `dim_store` and `dim_date`. No sales table, so no fan-out.*

In [10]:
# your query here
# q("""
#     SELECT ...
# """)

---
## Answers

**2.1**

In [11]:
q("""
SELECT s.market_name,
       ROUND(SUM(f.net_sales_gbp), 0) AS net_sales
FROM fact_sales f
JOIN dim_date d  ON f.date = d.full_date
JOIN dim_store s ON f.store_id = s.store_id
WHERE d.business_year = 2025
GROUP BY s.market_name
ORDER BY net_sales DESC
""")

,market_name,net_sales
0,United Kingdom,"14,751,294"
1,Germany,"5,602,809"
2,Ireland,"3,968,761"
3,Italy,"3,725,659"
4,Poland,"3,305,145"
5,Netherlands,"2,730,143"
6,Czechia,"1,136,120"
7,France,"1,091,173"
8,Latvia,"1,030,344"
9,Slovakia,"889,022"


**2.2**

In [12]:
q("""
SELECT p.gender, p.major_product_group,
       ROUND(SUM(f.net_sales_gbp), 0) AS net_sales
FROM fact_sales f
JOIN dim_date d    ON f.date = d.full_date
JOIN dim_product p ON f.sku = p.sku
WHERE d.business_year = 2025
GROUP BY p.gender, p.major_product_group
ORDER BY p.gender, net_sales DESC
""")

,gender,major_product_group,net_sales
0,Kids,Outerwear,"1,765,012"
1,Kids,Footwear,"1,157,179"
2,Kids,Tops,"853,926"
3,Kids,Legwear,"788,464"
4,Kids,Midlayer,"707,170"
...,...,...,...
20,Womens,Midlayer,"2,791,612"
21,Womens,Footwear,"1,611,343"
22,Womens,Legwear,"1,368,930"
23,Womens,Tops,"1,067,231"


**2.3**

In [13]:
q("""
SELECT pr.promo_name, pr.promo_type, SUM(f.quantity) AS units
FROM fact_sales f
JOIN dim_date d   ON f.date = d.full_date
JOIN dim_promo pr ON f.promo_id = pr.promo_id
WHERE d.business_year = 2025
  AND f.promo_id <> 'PROMO0000'
GROUP BY pr.promo_name, pr.promo_type
ORDER BY units DESC
LIMIT 5
""")

,promo_name,promo_type,units
0,Socks 3 for 2,Multi-buy,"68,608"
1,Summer Clearance 2025,Clearance,"44,136"
2,Summer Sale 2025,Seasonal Sale,"37,684"
3,Fleece & T-Shirts 2 for GBP30,Multi-buy,"33,473"
4,January Sale 2026,Seasonal Sale,"29,198"


**2.4**

In [14]:
q("""
SELECT s.market_name,
       ROUND(100.0 * SUM(ff.transactions) / SUM(ff.footfall), 1) AS conversion_pct
FROM fact_footfall ff
JOIN dim_store s ON ff.store_id = s.store_id
JOIN dim_date d  ON ff.date = d.full_date
WHERE d.business_year = 2025
GROUP BY s.market_name
ORDER BY conversion_pct DESC
""")

,market_name,conversion_pct
0,Italy,19.60
1,Netherlands,19.20
2,Germany,18.10
3,Ireland,17.70
4,United Kingdom,17.30
5,Latvia,16.90
6,Poland,16.50
7,Czechia,15.10
8,Slovakia,15
9,France,15
